# Upper Confidence Bound (UCB)

## The Problem: Where Should You Show Your Ad?

Imagine you are running a digital advertising campaign. You have **10 different ad designs** and you want to show users the one that gets clicked the most — but you do not know which ad is best at the start. You have two competing pressures:

- **Exploitation:** Show the ad that has worked best *so far*.
- **Exploration:** Try ads you have not shown much, in case one of them is actually much better.

This tension is called the **Exploration vs Exploitation dilemma**, and it appears in almost every sequential decision-making problem: recommendation systems, clinical trials, stock trading, and robot navigation.

---

## The Multi-Armed Bandit Framework

The classic framing is the **multi-armed bandit problem**:

> You are in a casino with 10 slot machines ("one-armed bandits"). Each machine pays out at a fixed but unknown probability. You have 10,000 pulls total. How do you maximise your total winnings?

In our case:
- Each **slot machine** = one ad design
- A **pull** = showing the ad to one user
- A **reward of 1** = the user clicked
- A **reward of 0** = the user did not click

---

## Why Not Just Test Each Ad Equally?

Pure **random exploration** wastes budget: you keep testing ads that are clearly bad. Pure **greedy exploitation** of the current best ad can lock you into a suboptimal choice early on.

UCB solves this by building a **confidence interval** around each ad's estimated click-through rate. Ads with fewer observations get a wider (higher) confidence bound — which gives them a chance to be selected and tested. As an ad gets selected more, its bound tightens to its true rate. You always pick the ad with the **highest upper confidence bound**.

---

## The UCB Formula

At round $n$, for ad $i$:

$$\text{UCB}_i(n) = \bar{r}_i + \sqrt{\frac{3}{2} \cdot \frac{\ln(n)}{N_i(n)}}$$

| Symbol | Meaning |
|--------|---------|
| $\bar{r}_i$ | Average reward (click rate) for ad $i$ so far |
| $n$ | Total number of rounds played |
| $N_i(n)$ | Number of times ad $i$ has been selected |
| $\ln(n)$ | Natural log of the total rounds — grows slowly, so the bonus shrinks over time |

**The bonus term** $\sqrt{\frac{3}{2} \cdot \frac{\ln(n)}{N_i(n)}}$ is large when ad $i$ has been selected few times and shrinks as $N_i$ grows. This naturally balances exploration and exploitation.

---

## What We Will Build

1. Simulate 10,000 users seeing ads from a dataset of 10 options
2. Use UCB to decide which ad to show at each step
3. Visualise which ad the algorithm converges on — and why that matters for a real campaign

## Step 1: Import Libraries

| Library | Why we need it |
|---------|---------------|
| `numpy` | Array operations and mathematical functions |
| `matplotlib` | Plotting the histogram of ad selections |
| `pandas` | Loading the dataset from CSV |
| `math` | `math.log()` and `math.sqrt()` for the UCB formula |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Step 2: Load the Dataset

The dataset `ads_ctr_optimization.csv` contains **10,000 rows and 10 columns**.

- Each **row** represents one user visiting the site.
- Each **column** represents one ad design (Ad 1 through Ad 10).
- A value of **1** means: *if this user had been shown this ad, they would have clicked.*
- A value of **0** means: *they would not have clicked.*

**Important:** In a real deployment, you only observe the reward for the one ad you actually showed. The other columns are hidden from you — the dataset is a simulation that lets us verify our algorithm found the best ad.

The task: using only the feedback column for the ad we select each round, find the best-performing ad as efficiently as possible.

In [ ]:
dataset = pd.read_csv('../data/ads_ctr_optimization.csv')

## Step 3: Implement the UCB Algorithm

This is the core of the notebook. Before reading the code, understand what we are tracking:

| Variable | Type | Purpose |
|----------|------|---------|
| `ads_selected` | list | History of which ad was shown at each round |
| `numbers_of_selections` | list of 10 ints | How many times each ad has been shown so far |
| `sums_of_rewards` | list of 10 ints | Total clicks accumulated for each ad |
| `total_reward` | int | Running total of all clicks — our performance metric |

**The algorithm at each round $n$:**

1. For each ad $i$, compute its UCB:
   - If it has been shown before: `average_reward + delta_i` (the UCB formula)
   - If it has **never** been shown: set the upper bound to infinity (`1e400`) — force the algorithm to try every ad at least once
2. Select the ad with the **highest upper bound**
3. Observe the reward (did this user click?)
4. Update `numbers_of_selections` and `sums_of_rewards` for the chosen ad

After all 10,000 rounds, `ads_selected` tells the complete story of which ad the algorithm preferred at each step.

In [ ]:
import math
N = 10000
d = 10
ads_selected = []
numbers_of_selections = [0] * d
sums_of_rewards = [0] * d
total_reward = 0
for n in range(0, N):
    ad = 0
    max_upper_bound = 0
    for i in range(0, d):
        if (numbers_of_selections[i] > 0):
            average_reward = sums_of_rewards[i] / numbers_of_selections[i]
            delta_i = math.sqrt(3/2 * math.log(n + 1) / numbers_of_selections[i])
            upper_bound = average_reward + delta_i
        else:
            upper_bound = 1e400
        if upper_bound > max_upper_bound:
            max_upper_bound = upper_bound
            ad = i
    ads_selected.append(ad)
    numbers_of_selections[ad] = numbers_of_selections[ad] + 1
    reward = dataset.values[n, ad]
    sums_of_rewards[ad] = sums_of_rewards[ad] + reward
    total_reward = total_reward + reward

## Step 4: Visualise the Results

The histogram shows **how many times each ad was selected** over 10,000 rounds.

**What to look for:**

- One ad should have a dramatically **taller bar** than all the others — this is the ad UCB identified as best.
- The other bars are short but non-zero — UCB explored all 10 ads early on, then committed to the winner.
- Compare the tallest bar to a uniform baseline of 1,000 (10,000 rounds / 10 ads). The more the algorithm converges, the taller the winner bar and the shorter the rest.

**Regret:** The ads that got many selections but were not the best represent *regret* — rounds where we could have shown the better ad. A good bandit algorithm minimises cumulative regret. UCB is proven to achieve **logarithmic regret** over time, meaning the per-round regret shrinks as the algorithm gets more confident.

**Why this matters for a real campaign:** In a 10,000-user campaign, the difference between showing a 2% CTR ad vs a 5% CTR ad is 300 extra clicks — or potentially hundreds of conversions and thousands in revenue. UCB finds the best ad *while running the campaign*, not after a separate A/B test.

In [ ]:
plt.hist(ads_selected)
plt.title('Histogram of ads selections')
plt.xlabel('Ads')
plt.ylabel('Number of times each ad was selected')
plt.show()